# `bz-ggml`: Dual-Instance Generator & Dedicated Streaming Vocoder on Dual Tesla T4 GPUs

This turnkey notebook demonstrates the high-performance C++ inference engine for **Breeze-TTS-2** powered by **GGML** on Dual NVIDIA Tesla T4 GPUs.

### Architecture Highlights:
- **GPU 0**: Text Encoder (prompt prefill) + **Generator Instance A** (Backbone + 15-step Depth Decoder in local VRAM).
- **GPU 1**: **Generator Instance B** (Backbone + 15-step Depth Decoder in local VRAM) + **Dedicated Streaming Neural Vocoder**.
- **Zero Intra-Frame PCIe Ping-Pong**: The heavy 15-pass Depth loop runs completely inside local GPU memory.
- **Immediate Frame 1 Audio**: Initial audio packet dispatches in **~229–280 ms TTFA**.
- **Sub-Real-Time Synthesis**: Single-stream RTF **0.686–0.750** on a single Tesla T4; cluster aggregate throughput **> 2.0x real-time**.

## 1. Hardware Verification
Confirm that two NVIDIA Tesla T4 GPUs (16 GB VRAM each) are detected.

In [ ]:
!nvidia-smi

## 2. Clone Repository & Build Native C++ Engine
We compile `libbreeze.so` with CUDA Compute Capability 75 (Turing T4) optimizations.

In [ ]:
import os

# If running directly inside the cloned repo, use current dir; otherwise clone
if not os.path.exists('/kaggle/working/bz-ggml'):
    !git clone --recursive https://github.com/sharjeel103/bz-ggml.git /kaggle/working/bz-ggml

%cd /kaggle/working/bz-ggml

# Build C++ core and shared library
!cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_BUILD_TYPE=Release
!cmake --build build --config Release -j$(nproc)

## 3. Download Quantized GGUF Model Weights
We use the official `breeze-tts-2-q8_0.gguf` (~4.8 GB INT8 quantized model).

In [ ]:
!mkdir -p /kaggle/working/models
model_path = '/kaggle/working/models/breeze-tts-2-q8_0.gguf'

if not os.path.exists(model_path):
    print('Downloading breeze-tts-2-q8_0.gguf...')
    !wget -c -O {model_path} https://huggingface.co/smcleod/Breeze-TTS-2-int8/resolve/main/breeze-tts-2-q8_0.gguf
    print('Download Complete!')
else:
    print('Model already downloaded at:', model_path)

## 4. Interactive Text-to-Speech Synthesis
Initialize the `DualInstanceCluster` across GPU 0 and GPU 1, synthesize speech, and listen to the audio directly in the notebook.

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/bz-ggml/python')

os.environ['LD_LIBRARY_PATH'] = (
    '/kaggle/working/bz-ggml/build/third_party/ggml/src:'
    '/kaggle/working/bz-ggml/build/third_party/ggml/src/ggml-cuda:'
    '/kaggle/working/bz-ggml/build:'
    '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
)

from bz_ggml import DualInstanceCluster, UserTask
import IPython.display as ipd

# Initialize Dual-Instance cluster (CUDA0 & CUDA1)
cluster = DualInstanceCluster('/kaggle/working/models/breeze-tts-2-q8_0.gguf')

prompt = "Speech synthesis with native C++ inference achieves sub-realtime latency and high throughput on modern accelerators."
task = UserTask(id=1, text=prompt, instruction="Speak clearly and naturally.")

results = cluster.run_workload([task], out_dir='/kaggle/working/interactive_audio')
res = results[0]

print(f"\n[Result] Synthesized {res.audio_s:.2f}s audio in {res.compute_wall_s:.2f}s wall time (RTF={res.rtf:.3f}, TTFA={res.ttfa_s:.3f}s)")
ipd.Audio(res.wav_path)

## 5. 40-Request Instant Burst Benchmark
Execute 40 real-world conversational sentences dispatched simultaneously at $t = 0.00\text{ s}$ with continuous 200 ms `nvidia-smi` hardware telemetry.

In [ ]:
!python3 /kaggle/working/bz-ggml/python/benchmark_40burst.py \
    --model /kaggle/working/models/breeze-tts-2-q8_0.gguf \
    --lib /kaggle/working/bz-ggml/build/libbreeze.so \
    --out-dir /kaggle/working/dual_gen_40burst_audio \
    --profile-csv /kaggle/working/dual_gen_40burst_profile.csv \
    --results-json /kaggle/working/dual_gen_40burst_results.json

## 6. Real-Time Hardware Telemetry Visualization
Plot GPU 0 vs GPU 1 compute utilization, power draw, and VRAM curves over the full 40-request burst test.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = '/kaggle/working/dual_gen_40burst_profile.csv'
if os.path.exists(csv_path):
    cols = ['timestamp', 'index', 'gpu_util', 'mem_util', 'power_w', 'vram_mb']
    df = pd.read_csv(csv_path, names=cols)
    df['gpu_util'] = pd.to_numeric(df['gpu_util'], errors='coerce')
    df['power_w'] = pd.to_numeric(df['power_w'], errors='coerce')
    df['vram_mb'] = pd.to_numeric(df['vram_mb'], errors='coerce')

    g0 = df[df['index'] == 0].reset_index(drop=True)
    g1 = df[df['index'] == 1].reset_index(drop=True)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # GPU Compute Utilization
    ax1.plot(g0.index * 0.2, g0['gpu_util'], label=f"GPU 0 (Instance A) Mean: {g0['gpu_util'].mean():.1f}%")
    ax1.plot(g1.index * 0.2, g1['gpu_util'], label=f"GPU 1 (Gen B + Vocoder) Mean: {g1['gpu_util'].mean():.1f}%")
    ax1.set_ylabel('Compute Utilization (%)')
    ax1.set_title('40-Request Burst Load: GPU Compute Utilization Over Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Power Draw
    ax2.plot(g0.index * 0.2, g0['power_w'], label=f"GPU 0 Power Mean: {g0['power_w'].mean():.1f} W")
    ax2.plot(g1.index * 0.2, g1['power_w'], label=f"GPU 1 Power Mean: {g1['power_w'].mean():.1f} W")
    ax2.set_xlabel('Elapsed Time (Seconds)')
    ax2.set_ylabel('Power Draw (Watts)')
    ax2.set_title('Power Draw (Card Envelope: 70W)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('Profile CSV not found at:', csv_path)